In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from apify_client import ApifyClient

# Load environment variables from .env file
load_dotenv()

# Get the API key safely from environment
APIFY_TOKEN = os.getenv("APIFY_TOKEN")

if not APIFY_TOKEN:
    raise ValueError("APIFY_TOKEN not found! Make sure it is defined in your .env file.")

# Initialize client using loaded token
client = ApifyClient(APIFY_TOKEN)

# Target handles for Super Bowl brands
brands = ["budlight", "budweiser", "pepsi", "doritos", "hyundai", "cocacola", "kia", "etrade", "toyota", "nfl"]

run_input = {
    "username": brands,
    "resultsLimit": 25,  # Fetch up to 25 individual posts per brand
}

print("Scraping raw post-level Instagram metrics...")
run = client.actor("apify/instagram-post-scraper").call(run_input=run_input)

# 2. Extract itemized post-level metrics
raw_posts = []
for item in client.dataset(run.default_dataset_id).iterate_items():
    raw_posts.append({
        "brand_handle": item.get("ownerUsername") or item.get("username"),
        "post_id": item.get("shortCode"),
        "post_url": f"https://www.instagram.com/p/{item.get('shortCode')}/" if item.get("shortCode") else None,
        "posted_at": item.get("timestamp"),
        "insta_likes_count": item.get("likesCount", 0),
        "insta_comments_count": item.get("commentsCount", 0),
        "caption": item.get("caption", "")[:150] if item.get("caption") else ""
    })

# 3. Create DataFrame and clean missing post metadata
insta_posts_df = pd.DataFrame(raw_posts).dropna(subset=["post_id"])

# 4. Map handles to Super Bowl brand names
brand_mapping = {
    "budlight": "Bud Light",
    "budweiser": "Budweiser",
    "pepsi": "Pepsi",
    "doritos": "Doritos",
    "hyundai": "Hynudai",  # Matches superbowl_ads_enriched.csv spelling
    "cocacola": "Coca-Cola",
    "kia": "Kia",
    "etrade": "E-Trade",
    "toyota": "Toyota",
    "nfl": "NFL"
}
insta_posts_df["brand"] = insta_posts_df["brand_handle"].map(brand_mapping)

# 5. Save exclusively as standalone CSV
insta_posts_df.to_csv("instagram_posts.csv", index=False)

print(f"Successfully generated 'instagram_posts.csv' with {len(insta_posts_df)} posts!")
print(insta_posts_df.head(10))

Scraping raw post-level Instagram metrics...


[apify.instagram-post-scraper runId:NldZzBSqTjS7sRhgk] -> Status: RUNNING, Message: 
[apify.instagram-post-scraper runId:NldZzBSqTjS7sRhgk] -> 2026-09-30T22:10:29.749Z ACTOR: Pulling container image of build UggZArVO8pclNEP4n from registry.
[apify.instagram-post-scraper runId:NldZzBSqTjS7sRhgk] -> 2026-09-30T22:10:29.752Z ACTOR: Creating container.
[apify.instagram-post-scraper runId:NldZzBSqTjS7sRhgk] -> 2026-09-30T22:10:29.813Z ACTOR: Starting container.
[apify.instagram-post-scraper runId:NldZzBSqTjS7sRhgk] -> 2026-09-30T22:10:29.816Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.instagram-post-scraper runId:NldZzBSqTjS7sRhgk] -> 2026-09-30T22:10:30.348Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.21.0"}
[apify.instagram-post-scraper runId:NldZzBSqTjS7sRhgk] -> 2026-09-30T22:10:30.521Z INFO  [Status message]: Starting the scraper with 10 direct URL(s)
[apify.instagram-post-scraper runId:Nl

Successfully generated 'instagram_posts.csv' with 154 posts!
           brand_handle      post_id  \
1                   nfl  Dd7KJEkGucO   
2                   nfl  Dd37sM8kRCR   
3   sisu.design.studios  C5NDln4Os85   
4   sisu.design.studios  C5m_7AauZo1   
5                   nfl  Dd442kyvGtS   
6                   nfl  Dd7FPz5GgDA   
7   sisu.design.studios  C5Px9-pO7gg   
8                   nfl  Dd6tsWfGQ-R   
9            tyler_loop  Dd7FHTOJ7_P   
10                  nfl  Dd67QqghFco   

                                    post_url                 posted_at  \
1   https://www.instagram.com/p/Dd7KJEkGucO/  2026-09-30T20:08:56.000Z   
2   https://www.instagram.com/p/Dd37sM8kRCR/  2026-09-29T14:04:56.000Z   
3   https://www.instagram.com/p/C5NDln4Os85/  2024-04-01T03:36:25.000Z   
4   https://www.instagram.com/p/C5m_7AauZo1/  2024-04-11T05:24:38.000Z   
5   https://www.instagram.com/p/Dd442kyvGtS/  2026-09-29T22:59:55.000Z   
6   https://www.instagram.com/p/Dd7FPz5GgDA/  2026-09-